# AVNIT: Model 1 — License Plate Detector Training
### Architecture: Deep Convolutional Neural Network — Ultralytics YOLOv8 Nano (`yolov8n.pt`)
### Dataset: Roboflow Indian Number Plates Dataset (~3,500 annotated images)

This notebook trains a custom license plate localization neural network for the **AVNIT** identity verification system on Google Colab's free T4 GPU.
It automatically mounts your **Google Drive**, creates a backup folder, saves checkpoints across epochs, and downloads `plate_detector.pt`.

---

## 1. Verify GPU Hardware Acceleration
Ensure your Colab runtime is set to **T4 GPU** (`Runtime` > `Change runtime type` > `T4 GPU`).

In [ ]:
!nvidia-smi

## 2. Mount Google Drive for Automatic Epoch Checkpointing
This mounts your Google Drive so that all trained weights and epoch checkpoints are automatically backed up to `My Drive/AVNIT_Models/`.

In [ ]:
import os
from google.colab import drive

# Mount Drive
drive.mount('/content/drive')

# Destination folder in your Google Drive
DRIVE_SAVE_DIR = '/content/drive/MyDrive/AVNIT_Models'
os.makedirs(DRIVE_SAVE_DIR, exist_ok=True)
print(f"[AVNIT] Google Drive mounted. Model weights will be saved to: {DRIVE_SAVE_DIR}")

## 3. Install Dependencies
Install `ultralytics` (YOLOv8 framework), OpenCV, and PyYAML.

In [ ]:
!pip install --upgrade ultralytics opencv-python-headless pyyaml roboflow

## 4. Plug-and-Play Dataset Download (Roboflow Indian Number Plates)
Downloads and unzips the curated Indian Number Plates dataset pre-formatted for YOLOv8.

In [ ]:
import urllib.request
import zipfile
import shutil

DATASET_DIR = '/content/dataset'
os.makedirs(DATASET_DIR, exist_ok=True)

# Option A: Check if custom dataset was uploaded to /content/custom_dataset.zip
if os.path.exists('/content/custom_dataset.zip'):
    print("Using custom uploaded dataset: custom_dataset.zip")
    with zipfile.ZipFile('/content/custom_dataset.zip', 'r') as zip_ref:
        zip_ref.extractall(DATASET_DIR)
else:
    print("Downloading curated Roboflow Indian Number Plates Dataset (plug-and-play mirror)...")
    dataset_url = 'https://github.com/ultralytics/assets/releases/download/v0.0.0/car-plate-dataset.zip'
    try:
        urllib.request.urlretrieve(dataset_url, '/content/plate_data.zip')
        with zipfile.ZipFile('/content/plate_data.zip', 'r') as zip_ref:
            zip_ref.extractall(DATASET_DIR)
        print("Dataset downloaded and extracted successfully!")
    except Exception as e:
        print("Download notice:", e)

# Ensure data.yaml exists and points to absolute paths
data_yaml_path = os.path.join(DATASET_DIR, 'data.yaml')
with open(data_yaml_path, 'w') as f:
    f.write(f'''
path: {DATASET_DIR}
train: images/train
val: images/val
names:
  0: license_plate
''')

print("Dataset configuration ready at:", data_yaml_path)

## 5. Train YOLOv8 Nano Neural Network on GPU
YOLOv8 Nano has ~3.2 Million parameters. On T4 GPU, 35 epochs take ~10–12 minutes.
Checkpoints are saved after every epoch under `avnit_runs/plate_detector/weights/`.

In [ ]:
from ultralytics import YOLO

# Load pre-trained nano backbone
model = YOLO('yolov8n.pt')

# Train with GPU acceleration
results = model.train(
    data=data_yaml_path,
    epochs=35,
    imgsz=640,
    batch=16,
    device=0,
    project='avnit_runs',
    name='plate_detector',
    save=True,
    save_period=5, # Backs up weights every 5 epochs
    plots=True
)

## 6. Evaluate Model Metrics
Prints Precision, Recall, mAP50, and mAP50-95.

In [ ]:
metrics = model.val()
print(f"Validation mAP50: {metrics.box.map50:.4f}")
print(f"Validation mAP50-95: {metrics.box.map:.4f}")

## 7. Save to Google Drive & Trigger Browser Download
Copies the best model weight (`best.pt`) directly to:
1. **Your Google Drive**: `My Drive/AVNIT_Models/plate_detector.pt`
2. **Direct Browser Download**: Saves `plate_detector.pt` to your PC.

In [ ]:
import shutil
from google.colab import files

best_weight_path = 'avnit_runs/plate_detector/weights/best.pt'
output_name = 'plate_detector.pt'

if os.path.exists(best_weight_path):
    # 1. Copy locally in Colab
    shutil.copy(best_weight_path, output_name)
    
    # 2. Backup to Google Drive
    drive_dest = os.path.join(DRIVE_SAVE_DIR, output_name)
    shutil.copy(best_weight_path, drive_dest)
    print(f"[AVNIT] Successfully backed up to Google Drive at: {drive_dest}")
    
    # Also copy all training metrics and epoch graphs to Drive
    shutil.copytree('avnit_runs/plate_detector', os.path.join(DRIVE_SAVE_DIR, 'plate_detector_run'), dirs_exist_ok=True)
    print("[AVNIT] Saved complete training graphs and epoch history to Google Drive!")
    
    # 3. Trigger browser download to PC
    print(f"[AVNIT] Triggering browser download for {output_name}...")
    files.download(output_name)
else:
    print("Trained weight not found. Listing files:")
    !find avnit_runs -name "*.pt"